In [ ]:
!lsof -ti:40006 | xargs kill -9

ERROR: The process "1234" not found.


In [1]:
# Install the scanner
!pip install pipreqsnb

# Run the scanner on your specific folder
# This scans ALL notebooks inside that folder and creates one requirements.txt
!pipreqsnb ../notebooks --force

pipreqs  --force ../notebooks
INFO: Not scanning for jupyter notebooks.
Please, verify manually the final list of requirements.txt to avoid possible dependency confusions.
Please, verify manually the final list of requirements.txt to avoid possible dependency confusions.
Please, verify manually the final list of requirements.txt to avoid possible dependency confusions.
Please, verify manually the final list of requirements.txt to avoid possible dependency confusions.
Please, verify manually the final list of requirements.txt to avoid possible dependency confusions.
Please, verify manually the final list of requirements.txt to avoid possible dependency confusions.
Please, verify manually the final list of requirements.txt to avoid possible dependency confusions.
Please, verify manually the final list of requirements.txt to avoid possible dependency confusions.
Please, verify manually the final list of requirements.txt to avoid possible dependency confusions.
Please, verify manually the 

In [2]:
import os
import yaml
with open("../config.yaml", "r") as f:
    config = yaml.safe_load(f)
GOOGLE_API_KEY = config['google']['api']
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["SERP_API_KEY"] = config['serp']['api']
os.environ["OPENAI_API_KEY"] = config['nautilus']['api']
os.environ["NRP_KEY"] = config['nautilus']['api']
os.environ["NRP_BASE"] = "https://ellm.nrp-nautilus.io/v1"

In [ ]:
import logging
import traceback
import uuid

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)
import gradio as gr
import re
import os
import sys
import spacy
import torch
import pandas as pd
import pickle
import requests
import json
import threading
import nest_asyncio
import uvicorn
import time
from rapidfuzz import fuzz
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from transformers import AutoTokenizer, AutoModelForSequenceClassification, BertModel
from google.adk.tools.agent_tool import AgentTool
from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm
from google.adk.a2a.utils.agent_to_a2a import to_a2a

sys.path.append('../utils/')
from mxnet_utils import BERTClassifier, CustomVocab

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
nlp = spacy.load("en_core_web_md")
analyzer = SentimentIntensityAnalyzer()

with open("../data/vocabs.pkl", "rb") as f:
    vocabs = pickle.load(f)

label_map = vocabs["label_map"]
topic_vocab = vocabs["topic_vocab"]
author_vocab = vocabs["author_vocab"]
job_vocab = vocabs["job_vocab"]
location_vocab = vocabs["location_vocab"]
affiliation_vocab = vocabs["affiliation_vocab"]

conservative_bigrams = pd.read_csv('../data/top_conservative_bigrams.csv')['bigram'].tolist()
liberal_bigrams = pd.read_csv('../data/top_liberal_bigrams.csv')['bigram'].tolist()

spam_tokenizer = AutoTokenizer.from_pretrained("mrm8488/bert-tiny-finetuned-sms-spam-detection")
spam_model = AutoModelForSequenceClassification.from_pretrained("mrm8488/bert-tiny-finetuned-sms-spam-detection")

bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert_base = BertModel.from_pretrained("bert-base-uncased")

net_best2 = BERTClassifier(
    bert=bert_base,
    num_topics=len(topic_vocab),
    num_authors=len(author_vocab),
    num_jobs=len(job_vocab),
    num_locations=len(location_vocab),
    num_affiliations=len(affiliation_vocab),
    num_classes=len(label_map),
    embed_dim=32,
    author_dropout=0.1,
    author_mlp_layers=2,
    author_mlp_hidden=192,
    history_dropout=0.1,
    history_mlp_layers=2,
    history_mlp_hidden=192,
)
net_best2.load_state_dict(torch.load("../checkpoints/best_cat.pth", map_location=device))
net_best2.to(device)
net_best2.eval()

def func_political_bias(text: str) -> str:
    statistic_types = {"CARDINAL", "PERCENT", "MONEY", "QUANTITY"}
    doc = nlp(str(text))
    stat_count = sum(ent.label_ in statistic_types for ent in doc.ents)

    def count_matches(stmt, bigram_list):
        words = [w.text.lower() for w in nlp(str(stmt))]
        if len(words) < 2:
            return 0
        bigrams = ["".join(words[i:i+2]) for i in range(len(words)-1)]
        matches = 0
        for bg in bigrams:
            for check in bigram_list:
                if fuzz.ratio(bg, check) >= 70:
                    matches += 1
                    break
        return matches

    return json.dumps({
        "stat_density": stat_count,
        "conservative_talking_points": count_matches(text, conservative_bigrams),
        "liberal_talking_points": count_matches(text, liberal_bigrams),
    })

def func_sensationalism(text: str) -> str:
    score = analyzer.polarity_scores(str(text))["compound"]
    return json.dumps({"emotional_intensity": abs(score), "polarity": score})

def func_spam(text: str) -> str:
    inputs = spam_tokenizer(str(text), return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = spam_model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)
    return json.dumps({"spam_probability": probs[0,1].item()})

def func_BERT(text: str) -> str:
    split_doc = nlp(str(text))
    sentences = [sent.text.strip() for sent in split_doc.sents if sent.text.strip()]
    rev = {v: k for k, v in label_map.items()}
    all_probs = []

    prob_list = []
    for sent in sentences:

        enc = bert_tokenizer(str(text), return_tensors="pt", truncation=True, max_length=512, padding=True).to(device)
        input_ids = enc["input_ids"]
        token_types = enc.get("token_type_ids", torch.zeros_like(input_ids)).to(device)
        mask = enc["attention_mask"]

        with torch.no_grad():
            outputs = net_best2(
                input_ids,
                token_types,
                mask,
                torch.zeros(1, len(topic_vocab)).to(device),
                torch.tensor([0]).to(device),
                torch.tensor([0]).to(device),
                torch.tensor([0]).to(device),
                torch.tensor([0]).to(device),
                torch.zeros(1, len(label_map)).to(device),
            )
            probs = torch.softmax(outputs, dim=1)
            prob_list.append(probs[0].cpu()) 
            
    avg_probs = torch.stack(prob_list).mean(dim=0)
    pred = torch.argmax(avg_probs).item()

    return json.dumps({
        "model_prediction": rev[pred],
        "confidence": probs[0,pred].item(),
        "class_probabilities": {rev[i]: probs[0,i].item() for i in range(len(label_map))}
    })

def func_web_search(text: str) -> str:
    url = "https://serpapi.com/search"
    params = {"q": text, "api_key": os.environ.get("SERP_API_KEY"), "engine": "google", "num": 4}
    try:
        res = requests.get(url, params=params).json()
        evidence = []
        if "answer_box" in res:
            evidence.append(res["answer_box"].get("answer") or res["answer_box"].get("snippet"))
        for item in res.get("organic_results", []):
            evidence.append(item.get("snippet"))
        return "\n".join(evidence) if evidence else "No live evidence found."
    except:
        return "Search Error"

worker_llm = LiteLlm(model="gemini/gemini-3-pro-preview", api_key=os.environ.get("GOOGLE_API_KEY"))
manager_llm = LiteLlm(model="gemini/gemini-3-pro-preview", api_key=os.environ.get("GOOGLE_API_KEY"))

worker_instruction = """
1. Call your assigned tool immediately with the text you receive.
2. Review the data returned by your tool.
3. Return a SINGLE valid JSON object with EXACTLY two keys:
   - "tool_output": The raw data, metrics, or search results returned by your tool.
   - "explanation": A concise, 1-2 sentence explanation of what those results mean in the context of the text.
   
CRITICAL: Output ONLY valid JSON. Do not include markdown code blocks (like ```json), and do not add any text outside the JSON object.
"""

specialist_instruction = """
You are a specialist analyst. Follow these steps EXACTLY:

STEP 1: Call your assigned tool immediately with the EXACT text you receive as input. Do not paraphrase it.
STEP 2: Wait for the tool to return its output.
STEP 3: Analyze what the tool output means in the context of the input text and its truthfulness.
STEP 4: Return your response as a SINGLE valid JSON object with EXACTLY these two keys:
   - "raw_data": The complete, unmodified output returned by your tool (as a parsed object, not a string).
   - "explanation": A 2-3 sentence analysis of what those results reveal about the factual reliability of the input text.

CRITICAL RULES:
- Output ONLY valid JSON. No markdown. No code fences. No text before or after the JSON object.
- You MUST call your tool before responding. Do NOT fabricate tool output.
- The "explanation" must reference specific values from "raw_data" (e.g. scores, probabilities, counts).
- Do NOT return a generic response. Tie your explanation directly to the numbers your tool returned.

Example output format:
{"raw_data": {"some_score": 0.87}, "explanation": "The tool returned a score of 0.87, indicating high likelihood of X, which suggests the input text is Y because Z."}
"""

bias_agent = LlmAgent(name="Political_Bias_Agent", model=worker_llm, instruction=specialist_instruction, tools=[func_political_bias])
sensational_agent = LlmAgent(name="Sensationalism_Agent", model=worker_llm, instruction=specialist_instruction, tools=[func_sensationalism])
spam_agent = LlmAgent(name="Spam_Agent", model=worker_llm, instruction=specialist_instruction, tools=[func_spam])
bert_agent = LlmAgent(name="BERT_Agent", model=worker_llm, instruction=specialist_instruction, tools=[func_BERT])
search_agent = LlmAgent(name="Web_Search_Agent", model=worker_llm, instruction=worker_instruction, tools=[func_web_search])

manager_prompt = """
CRITICAL:
Your final message MUST be ONLY valid JSON.
Do not include markdown.
Do not include explanations outside JSON.
Do not include code fences.
You are the Factuality Root Manager. Your job is to generate a final fact-checking JSON report.

###CRITICAL: MULTI-TURN PERSISTENCE
USE MULTI-TURN PERSISTENCE IF AGENT OUTPUTS ARE EMPTY OR INVALID!
Updated Manager Operational Protocol

PHASE 1: MANDATORY STATE INITIALIZATION

Before taking any action, initialize your Internal Evidence Ledger. You are forbidden from providing a veracity_label until every slot in this ledger is marked [COMPLETE].



[Slot A] BERT_Agent: Truthfulness probabilities and model confidence.

[Slot B] Bias_Agent: Statistical density and partisan bigram counts.

[Slot C] Sensationalism_Agent: Emotional intensity and polarity scores.

[Slot D] Spam_Agent: Bot-likelihood and structural repetition check.

[Slot E] Search_Agent: External corroboration of specific claims.

PHASE 2: SEQUENTIAL EXECUTION & PERSISTENCE

You must call agents one by one. For each turn:



Evaluate: Look at your history. Which Ledger Slots are still [EMPTY]?

Execute: Call the agent corresponding to the first empty slot.

Synthesize: When the agent responds, summarize their "explanation" field into your ledger.

Persistence Rule: If an agent returns an error or an empty string, you must re-attempt the call once using a more specific search query or a different segment of the article text.

PHASE 3: CROSS-FIELD VALIDATION

Once the Ledger is [COMPLETE], perform a final consistency check:



Does the Search_Agent evidence support the BERT_Agent probability?

Does high Sensationalism correlate with high Political Bias?

If contradictions exist, mention them explicitly in your final explanation_text.

### CRITICAL: MANDATORY DATA GATHERING
You MUST consult your sub-agents to gather data BEFORE generating your final response to the user. Do not stop or reply to the user until you have collected sufficient information from AT LEAST ONE of the following agents:

1. Call 'BERT_Agent' to get truthfulness probabilities.
2. Call 'Political_Bias_Agent' to check for stats and partisan framing.
3. Call 'Sensationalism_Agent' to get the emotional intensity score.
4. Call 'Spam_Agent' to check for bot-like characteristics.
5. Call 'Web_Search_Agent' to verify the text. You MUST pass a specific search query containing the core claim, names, or keywords extracted from the user's article. Do NOT pass literal phrases from these instructions like "live evidence".
### TOOL CALL RULES:
- When calling Web_Search_Agent, do NOT pass the entire article. 
- Instead, extract the 3 most important entities (people, places, dates) and the primary claim.
- Format the tool input as a search query: "[Primary Claim] + [Entity 1] + [Entity 2]".

Wait for each agent to return its data, keep it in your internal memory, and immediately call the next agent on the list.

### FINAL SYNTHESIS
ONLY AFTER you have received data from all 6 agents, synthesize the results and output the final report.

[ORIGINAL FACTUALITY INSTRUCTIONS]
You are an AI assistant assigned to evaluate the factuality of news statements using a generative fact-checking pipeline.
Your task is to analyze article text, incorporate predictive model outputs WITHOUT overweighting them, and compute factor scores using the scoring recipes below.

VECTOR DESCRIPTION:
0-5: Probabilities for truthfulness classes (BERT_Agent).
7: Count of numeric/statistical entities.
8: Count of conservative bigram matches.
9: Count of liberal bigram matches.
10: Emotional intensity score (Sensationalism_Agent).
11: Spam likelihood score (Spam_Agent).

ANTI-BIAS CONSTRAINT:
- Treat predictive model scores only as auxiliary context. Rely on TEXTUAL EVIDENCE for final verdicts.

FACTUALITY FACTORS:
1. AUTHENTICITY (1–10): Verifiable details, sources, timestamps.
2. SENSATIONALISM (1–10): Density of hyperbole/drama.
3. POLITICAL BIAS (0–10 + tag): Partisan framing/selective omission.
4. SPAM (1–10): Bot-like content characteristics.
5. CONFIRMATION BIAS (1–10): Cherry-picked evidence.
6. SHORT-TERM UTILITY (1–10): Clickbait or monetization cues.

OUTPUT FORMAT (STRICT JSON):
{
  "veracity_label": "One of: True, Mostly True, Half True, Mostly False, False, Pants on Fire",
  "explanation_text": "...",
  "factor_scores": [
    {"factor": "Authenticity", "score": 1-10, "reasoning": "..."},
    ...
  ]
}
"""

manager_agent = LlmAgent(
    name="FactCheck_Manager",
    model=manager_llm,
    instruction=manager_prompt,
    tools=[
        AgentTool(agent=bert_agent),
        AgentTool(agent=bias_agent),
        AgentTool(agent=sensational_agent),
        AgentTool(agent=spam_agent),
        AgentTool(agent=search_agent),
    ],
)

nest_asyncio.apply()
a2a_app = to_a2a(manager_agent)

def run_server():
    uvicorn.run(a2a_app, host="0.0.0.0", port=40008, log_level="error")

threading.Thread(target=run_server, daemon=True).start()
time.sleep(2)

def call_agent_api(text: str) -> str:
    base_url = "http://localhost:40008/"

    send_payload = {
        "jsonrpc": "2.0",
        "method": "message/send",
        "id": 1,
        "params": {
            "message": {
                "messageId": str(uuid.uuid4()),  # only messageId, NO taskId
                "role": "user",
                "parts": [{"kind": "text", "text": text}]
            }
        }
    }

    try:
        logger.info("Sending request")
        r = requests.post(base_url, json=send_payload, timeout=180)
        logger.info(f"Status: {r.status_code}")
        r.raise_for_status()

        j = r.json()
        logger.info(f"Raw response:\n{json.dumps(j, indent=2)}")

        if "error" in j:
            logger.error(f"JSON-RPC error: {j['error']}")
            return ""

        return extract_text_from_a2a_response_v03(j)

    except requests.exceptions.Timeout:
        logger.error("Request timed out.")
        return ""
    except Exception as e:
        logger.error(f"call_agent_api failed:\n{traceback.format_exc()}")
        return ""

def extract_text_from_a2a_response_v03(j: dict) -> str:
    """Parse A2A v0.3.0 response envelope."""
    result = j.get("result", {})

    for part in result.get("parts", []):
        if part.get("kind") == "text" and part.get("text"):
            return part["text"]

    for artifact in result.get("artifacts", []):
        for part in artifact.get("parts", []):
            if part.get("kind") == "text" and part.get("text"):
                return part["text"]

    status_msg = result.get("status", {}).get("message", {})
    for part in status_msg.get("parts", []):
        if part.get("kind") == "text" and part.get("text"):
            return part["text"]

    logger.warning(f"Could not extract text. Full result:\n{json.dumps(result, indent=2)}")
    return ""
def run_full_pipeline_agentic(article_text, progress=gr.Progress()):
    if not article_text.strip():
        return "Error", [], "Empty input", {}

    bert_out = json.loads(func_BERT(article_text))
    bias_out = json.loads(func_political_bias(article_text))
    sent_out = json.loads(func_sensationalism(article_text))
    spam_out = json.loads(func_spam(article_text))

    class_probs = bert_out.get("class_probabilities", {})

    vector = {
        "bert_true":           class_probs,
        "stat_entity_count":    bias_out.get("stat_density", 0),
        "conservative_bigrams": bias_out.get("conservative_talking_points", 0),
        "liberal_bigrams":      bias_out.get("liberal_talking_points", 0),
        "emotional_intensity":  sent_out.get("emotional_intensity", 0),
        "spam_probability":     spam_out.get("spam_probability", 0),
    }

    resp = call_agent_api(article_text)
    if not resp:
        return "Error", [], "Agent returned empty response", vector

    m = re.search(r"\{.*\}", resp, re.DOTALL)
    if not m:
        return "Error", [], resp, vector

    data = json.loads(m.group(0))
    veracity    = data.get("veracity_label", "Unknown")
    explanation = data.get("explanation_text", "")
    scores      = data.get("factor_scores", [])
    df          = [[s["factor"], s["score"], s["reasoning"]] for s in scores]

    return veracity, df, explanation, vector

with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue")) as demo:
    gr.Markdown("# Agentic News Fact-Analysis")

    with gr.Row():
        with gr.Column():
            article_input = gr.Textbox(lines=15, label="Input Article")
            btn = gr.Button("Run Analysis")

        with gr.Column():
            verdict = gr.Label(label="Manager Verdict")
            summary = gr.Markdown()
            table = gr.Dataframe(headers=["Factor", "Score", "Reasoning"])
            meta = gr.JSON()

    btn.click(run_full_pipeline_agentic, article_input, [verdict, table, summary, meta])

demo.queue()
demo.launch(share=True)

/var/folders/1q/y24s3byn429835g67zr653980000gn/T/ipykernel_2673/695316635.py:162: UserWarning: [GEMINI_VIA_LITELLM] gemini/gemini-3-pro-preview: You are using Gemini via LiteLLM. For better performance, reliability, and access to latest features, consider using Gemini directly through ADK's native Gemini integration. Replace LiteLlm(model='gemini/gemini-3-pro-preview') with Gemini(model='gemini-3-pro-preview'). Set ADK_SUPPRESS_GEMINI_LITELLM_WARNINGS=true to suppress this warning.
  worker_llm = LiteLlm(model="gemini/gemini-3-pro-preview", api_key=os.environ.get("GOOGLE_API_KEY"))
/var/folders/1q/y24s3byn429835g67zr653980000gn/T/ipykernel_2673/695316635.py:163: UserWarning: [GEMINI_VIA_LITELLM] gemini/gemini-3-pro-preview: You are using Gemini via LiteLLM. For better performance, reliability, and access to latest features, consider using Gemini directly through ADK's native Gemini integration. Replace LiteLlm(model='gemini/gemini-3-pro-preview') with Gemini(model='gemini-3-pro-preview

* Running on local URL:  http://127.0.0.1:7872


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


* Running on public URL: https://360474d96a7d0f3160.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


2026-03-09 16:10:28,914 - INFO - Sending request
2026-03-09 16:10:42,949 - INFO - Status: 200
2026-03-09 16:10:42,949 - INFO - Raw response:
{
  "id": 1,
  "jsonrpc": "2.0",
  "result": {
    "artifacts": [
      {
        "artifactId": "47b86e50-a28b-476b-9193-b9ed19bdf2fb",
        "parts": [
          {
            "kind": "text",
            "text": "{\n  \"veracity_label\": \"Pants on Fire\",\n  \"explanation_text\": \"The claim is completely false. LeBron James is a professional basketball player, not the President of the United States. As of January 2025, the President of the United States is Donald Trump.\",\n  \"factor_scores\": [\n    {\n      \"factor\": \"Authenticity\",\n      \"score\": 1,\n      \"reasoning\": \"The statement contains zero verifiable facts aligning with reality. LeBron James has never held public office.\"\n    },\n    {\n      \"factor\": \"Sensationalism\",\n      \"score\": 2,\n      \"reasoning\": \"The claim itself is a simple, brief sentence withou

In [41]:
func_BERT("""WASHINGTON, D.C. — With military operations ongoing against the Ayatollah's Islamic regime, former President Barack Obama expressed confusion at seeing bombs falling on Iran instead of pallets stacked with U.S. cash.

The former American leader reportedly told his security staff that it was strange to see news footage of explosive ordnance falling from the sky to destroy Tehran when he was used to seeing astronomical amounts of U.S. currency being parachuted into the America-hating country.

"That's not what I'm used to," Obama said. "Did anyone let Trump know that the United States is supposed to be dropping tons and tons of cash on Iran, and not bombs? Someone might want to get word to him that the Ayatollah appreciates U.S. dollars, and not cruise missiles. I knew Trump had no idea how the world really worked, but man, dropping bombs on Iran? That's not how I used to do things."

Sources within Obama's household staff said the former president was confused by the whole situation. "I'm not sure what Trump thinks he's going to accomplish with this," he reportedly said. "How does he expect to fund the Iranian nuclear weapons program and global terror network with bombs and missiles? It doesn't make sense. He's in danger of wiping out the entire regime by doing this, which is the opposite of what I was trying to do when I was in office. It's a shame."

At publishing time, Obama had also expressed confusion about the U.S. providing security to its embassy and consulate locations instead of allowing them to be overrun and destroyed by radical Muslim terrorists.
""")

'{"model_prediction": "false", "confidence": 0.2143004983663559, "class_probabilities": {"pants-fire": 0.1909695416688919, "false": 0.2143004983663559, "barely-true": 0.16630147397518158, "half-true": 0.14415596425533295, "mostly-true": 0.18598847091197968, "true": 0.09828396886587143}}'

In [42]:
func_BERT("""
The Mazda MX-5 is a lightweight two-seat sports car manufactured and marketed by Mazda. In Japan, it is marketed as the Mazda Roadster[a] or, previously, as the Eunos Roadster.[b] In the United States it is sold as the Mazda Miata (/miˈɑːtə/), and it was formerly marketed under the same name in Canada. The name miata derives from Old High German for "reward".[c]

Produced at Mazda's Hiroshima plant, the MX-5 debuted in 1989 at the Chicago Auto Show. It was created under the design credo Jinba ittai, meaning "unity of horse and rider". Noted for its small, light, balanced and minimalist design, the MX-5 has often been described as a successor to the 1950s and 1960s Italian and British roadsters, with the Lotus Elan serving as a design benchmark.

Each generation is identified by a two-letter code, beginning with the first generation NA. The second generation NB launched in 1998, followed by the third generation NC in 2005, and the fourth generation ND in 2015.

More than one million MX-5s have been sold, making it the best-selling two-seat convertible sports car in history""")

'{"model_prediction": "mostly-true", "confidence": 0.2570079267024994, "class_probabilities": {"pants-fire": 0.1178128570318222, "false": 0.17575354874134064, "barely-true": 0.18073494732379913, "half-true": 0.21264997124671936, "mostly-true": 0.2570079267024994, "true": 0.056040793657302856}}'

https://fff8facc50b59a8302.gradio.live/